# Impacto consolidado del graduado USTA — **SECOP + RUES + CvLAC**

**Objetivo.** Unir en **una sola tabla por persona (cédula)** las tres dimensiones de
impacto del egresado construidas en los cuadernos previos, para responder *quién* deja
huella y *en cuántas* dimensiones a la vez.

| Dimensión | Fuente | Señal | Insumo |
|-----------|--------|-------|--------|
| **Contratación pública** | SECOP Integrado | `es_proveedor_secop` | `graduados_proveedores_secop.csv` |
| **Emprendimiento** | RUES (Confecámaras) | `en_rues` / empresa activa | `graduados_emprendedores_rues.csv` |
| **Investigación** | CvLAC (ScienTI) | `en_cvlac` (validado por evidencia USTA) | `graduados_cvlac_cruce.csv` |

> **Llave de unión:** la **cédula** (`identificacion`). SECOP y RUES cruzan por documento;
> CvLAC se validó por nombre + evidencia USTA y aquí se atribuye a la cédula del graduado.
> Se construye **una fila por cédula** (persona), agregando sus programas/sedes/fuentes.


## 1. Configuración e insumos

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
SALIDA = Path("salidas")

req = ["graduados_integrado.csv", "graduados_proveedores_secop.csv",
       "graduados_emprendedores_rues.csv", "graduados_cvlac_cruce.csv"]
for f in req:
    p = SALIDA / f
    print(("OK " if p.exists() else "FALTA"), p)


## 2. Base de personas (una fila por cédula)

Del dataset integrado se deduplica por cédula, conservando un nombre representativo y
**agregando** los programas, sedes y fuentes asociadas a esa persona.


In [ ]:
grad = pd.read_csv(SALIDA / "graduados_integrado.csv", dtype={"identificacion": "string"})
grad["identificacion"] = grad["identificacion"].str.strip()
g = grad[grad["identificacion"].notna() & (grad["identificacion"].str.len() > 0)].copy()

def join_unicos(s):
    vals = [str(x) for x in s.dropna().unique() if str(x).strip()]
    return " | ".join(sorted(vals))

base = (g.groupby("identificacion")
        .agg(nombre=("nombre_completo", "first"),
             programas=("programa", join_unicos),
             sedes=("sede", join_unicos),
             fuentes=("fuente", join_unicos),
             n_titulos=("programa", "size"),
             anio_primer_grado=("anio_grado", "min"),
             anio_ultimo_grado=("anio_grado", "max"))
        .reset_index())
print(f"Personas (cédulas únicas): {len(base):,}")
base.head(3)


## 3. Dimensión SECOP (contratación pública)

In [ ]:
secop = pd.read_csv(SALIDA / "graduados_proveedores_secop.csv", dtype={"identificacion": "string"})
secop_x = (secop.dropna(subset=["identificacion"])
           .drop_duplicates("identificacion")
           [["identificacion", "n_contratos", "valor_total", "ultima_firma"]]
           .rename(columns={"valor_total": "secop_valor_total",
                            "n_contratos": "secop_n_contratos",
                            "ultima_firma": "secop_ultima_firma"}))
base = base.merge(secop_x, on="identificacion", how="left")
base["es_proveedor_secop"] = base["secop_n_contratos"].notna()
print("Proveedores SECOP:", int(base["es_proveedor_secop"].sum()))


## 4. Dimensión RUES (emprendimiento)

In [ ]:
rues = pd.read_csv(SALIDA / "graduados_emprendedores_rues.csv", dtype={"identificacion": "string"})
rues_x = (rues.dropna(subset=["identificacion"])
          .drop_duplicates("identificacion")
          [["identificacion", "n_matriculas", "n_activas", "tiene_empresa_activa",
            "ciiu_principal", "ultima_matricula"]]
          .rename(columns={"n_matriculas": "rues_n_matriculas",
                           "n_activas": "rues_n_activas",
                           "ciiu_principal": "rues_ciiu",
                           "ultima_matricula": "rues_ultima_matricula"}))
base = base.merge(rues_x, on="identificacion", how="left")
base["en_rues"] = base["rues_n_matriculas"].notna()
base["rues_empresa_activa"] = base["tiene_empresa_activa"].fillna(False).astype(bool)
print("Emprendedores RUES:", int(base["en_rues"].sum()),
      "| con empresa activa:", int(base["rues_empresa_activa"].sum()))


## 5. Dimensión CvLAC (investigación)

CvLAC se validó por persona/nombre; aquí se agrega **por cédula**: una cédula está en
CvLAC si **alguno** de sus registros quedó validado (`validado_cvlac`). Se toma el de mayor
score como representante.


In [ ]:
cvl = pd.read_csv(SALIDA / "graduados_cvlac_cruce.csv", dtype={"identificacion": "string"})
cvl["identificacion"] = cvl["identificacion"].str.strip()
val = cvl[(cvl["validado_cvlac"] == True) & cvl["identificacion"].notna()].copy()
cvl_x = (val.sort_values("score", ascending=False)
         .drop_duplicates("identificacion")
         [["identificacion", "cod_rh", "nombre_cvlac", "nivel_maximo",
           "categoria_minciencias", "total_productos", "score", "en_grupo_usta"]]
         .rename(columns={"total_productos": "cvlac_productos",
                          "score": "cvlac_score",
                          "nivel_maximo": "cvlac_nivel",
                          "categoria_minciencias": "cvlac_categoria"}))
base = base.merge(cvl_x, on="identificacion", how="left")
base["en_cvlac"] = base["cod_rh"].notna()
print("Investigadores CvLAC (validados):", int(base["en_cvlac"].sum()))


## 6. Índice de impacto multidimensional

Se cuenta en **cuántas** de las tres dimensiones aparece cada persona.


In [ ]:
base["n_dimensiones"] = (base["es_proveedor_secop"].astype(int)
                          + base["en_rues"].astype(int)
                          + base["en_cvlac"].astype(int))

def perfil(r):
    p = []
    if r["es_proveedor_secop"]: p.append("SECOP")
    if r["en_rues"]: p.append("RUES")
    if r["en_cvlac"]: p.append("CvLAC")
    return "+".join(p) if p else "(ninguna)"
base["perfil_impacto"] = base.apply(perfil, axis=1)

N = len(base)
print(f"Total personas: {N:,}\n")
print("Distribución por nº de dimensiones de impacto:")
print(base["n_dimensiones"].value_counts().sort_index().to_string())
print(f"\nCon al menos una dimensión: {int((base['n_dimensiones'] >= 1).sum()):,} "
      f"({(base['n_dimensiones'] >= 1).mean()*100:.1f}%)")
print(f"En las tres a la vez       : {int((base['n_dimensiones'] == 3).sum()):,}")


## 7. Análisis

### 7.1 Cobertura por dimensión y combinaciones


In [ ]:
dims = pd.Series({
    "SECOP (contratación)": int(base["es_proveedor_secop"].sum()),
    "RUES (emprendimiento)": int(base["en_rues"].sum()),
    "CvLAC (investigación)": int(base["en_cvlac"].sum()),
})
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
dims.plot(kind="bar", ax=axes[0], color=["#274690", "#1B998B", "#5C415D"])
axes[0].set_title("Personas por dimensión de impacto"); axes[0].tick_params(axis="x", rotation=15)
combo = base[base["n_dimensiones"] >= 1]["perfil_impacto"].value_counts()
combo.sort_values().plot(kind="barh", ax=axes[1], color="#E07A5F")
axes[1].set_title("Combinaciones de impacto (personas con ≥1)")
plt.tight_layout(); plt.show()
display(combo.to_frame("personas"))


### 7.2 Programas con mayor impacto multidimensional

In [ ]:
# Atribución por programa: se reparte cada persona a los programas que cursó
expl = base.assign(programa=base["programas"].str.split(" | ", regex=False)).explode("programa")
expl["programa"] = expl["programa"].str.strip()
prog = (expl.groupby("programa")
        .agg(personas=("identificacion", "nunique"),
             secop=("es_proveedor_secop", "sum"),
             rues=("en_rues", "sum"),
             cvlac=("en_cvlac", "sum"),
             multi=("n_dimensiones", lambda s: int((s >= 2).sum())))
        .sort_values("personas", ascending=False).head(15))
display(prog)


### 7.3 Personas con impacto en las tres dimensiones

In [ ]:
tri = (base[base["n_dimensiones"] == 3]
       .sort_values(["cvlac_score", "secop_valor_total"], ascending=False)
       [["identificacion", "nombre", "programas", "secop_n_contratos", "secop_valor_total",
         "rues_n_matriculas", "rues_empresa_activa", "cvlac_nivel", "cvlac_categoria",
         "cvlac_productos"]])
print(f"Personas en SECOP + RUES + CvLAC: {len(tri):,}")
display(tri.head(25))


## 8. Exportación

- `impacto_consolidado.csv`: **una fila por cédula** con las tres dimensiones y el índice.


In [ ]:
ruta = SALIDA / "impacto_consolidado.csv"
orden = ["identificacion", "nombre", "programas", "sedes", "fuentes", "n_titulos",
         "anio_primer_grado", "anio_ultimo_grado",
         "es_proveedor_secop", "secop_n_contratos", "secop_valor_total", "secop_ultima_firma",
         "en_rues", "rues_n_matriculas", "rues_n_activas", "rues_empresa_activa", "rues_ciiu",
         "en_cvlac", "cod_rh", "cvlac_nivel", "cvlac_categoria", "cvlac_productos",
         "cvlac_score", "en_grupo_usta",
         "n_dimensiones", "perfil_impacto"]
base[orden].to_csv(ruta, index=False, encoding="utf-8-sig")
print("Exportado:", ruta.resolve(), f"({len(base):,} personas)")


## 9. Conclusiones

- Se consolidó el impacto del egresado en **una tabla por persona** uniendo tres registros
  oficiales: **SECOP** (proveedor del Estado), **RUES** (emprendimiento) y **CvLAC**
  (investigación, validado con evidencia USTA).
- El **índice `n_dimensiones`** resume en cuántas facetas deja huella cada graduado; el
  campo `perfil_impacto` indica la combinación exacta (p. ej. `RUES+CvLAC`).

**Limitaciones heredadas de cada cruce**
- **SECOP/RUES**: match por cédula (no captura sociedades por NIT en RUES; ver cuadernos).
- **CvLAC**: vínculo por nombre + evidencia USTA y **universo parcial**; subestima.
- La unión por cédula depende de la calidad del documento en cada fuente.
